In [1]:
import chess, chess.engine, os, stat
from policy import *
import random
from discrim import *
from file_helper import truncateGame

2026-04-23 00:22:31.287211: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-23 00:22:31.332586: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-23 00:22:33.002840: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
/storage/icds/RISE/sw8/anaconda/conda_envs/pytorch/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.24.3
  warnings.warn(f"A NumPy version 

POLICY V7


In [2]:
from stockfish import Stockfish
engine_path = r"./stockfish/src/stockfish"
sf = Stockfish(engine_path, parameters={"Threads": 1, "Hash": 256})
sf.set_depth(2)
sf.set_skill_level(2)
sf.get_engine_parameters()

{'Debug Log File': '',
 'Threads': 1,
 'Hash': 256,
 'Ponder': False,
 'MultiPV': 1,
 'Skill Level': 2,
 'Move Overhead': 10,
 'Slow Mover': 100,
 'UCI_Chess960': False,
 'UCI_LimitStrength': False,
 'UCI_Elo': 1350,
 'Contempt': 0,
 'Min Split Depth': 0,
 'Minimum Thinking Time': 20}

In [3]:
games= load_json("./data/Bijay_1549_games.json")
print(len(games))

Loading games: 100%|██████████| 392/392 [00:00<00:00, 517.79it/s]

392


In [4]:
agent = Agent("Bijay_1549",stockfish_path=r"./stockfish/src/stockfish")
agent.train(games)

Agent V2
[]
Epoch 1/10
396/396 [==============================] - 127s 313ms/step - loss: 0.0027 - accuracy: 0.0852
Epoch 2/10
396/396 [==============================] - 124s 314ms/step - loss: 0.0019 - accuracy: 0.0928
Epoch 3/10
396/396 [==============================] - 126s 319ms/step - loss: 0.0016 - accuracy: 0.0925
Epoch 4/10
396/396 [==============================] - 125s 316ms/step - loss: 0.0014 - accuracy: 0.0923
Epoch 5/10
396/396 [==============================] - 125s 317ms/step - loss: 0.0011 - accuracy: 0.0912
Epoch 6/10
396/396 [==============================] - 123s 312ms/step - loss: 8.0707e-04 - accuracy: 0.0912
Epoch 7/10
396/396 [==============================] - 125s 317ms/step - loss: 5.7273e-04 - accuracy: 0.0915
Epoch 8/10
396/396 [==============================] - 126s 318ms/step - loss: 3.9390e-04 - accuracy: 0.0919
Epoch 9/10
396/396 [==============================] - 124s 313ms/step - loss: 2.7749e-04 - accuracy: 0.0933
Epoch 10/10
396/396 [===============

In [5]:
import os

def simulate_games(agent, sf, num_games=400, file_dir="./data", file_postfix="0"):
    os.makedirs(file_dir, exist_ok=True)  # ✅ fix: create dir before writing
    
    games_data = []
    i = 0
    while i < num_games:
        board = chess.Board()
        moves = []
        aborted = False
        
        while not board.is_game_over():
            try:
                if board.turn == chess.WHITE:
                    move = agent.act(board)
                else:
                    sf.set_fen_position(board.fen())
                    best = sf.get_best_move()
                    if best is None:
                        aborted = True
                        break
                    move = chess.Move.from_uci(best)
            except Exception as e:
                print(f"[Game {i+1}] Error: {e}")
                aborted = True
                break
            
            board.push(move)
            moves.append(move.uci())
        
        if aborted:
            continue  # ✅ fix: skip broken games instead of saving them
        
        game_data = {
            "event": "Agent vs Stockfish",
            "round": i + 1,
            "white": f"Mimic Agent of {agent.id}",
            "black": "Stockfish",
            "result": board.result(),
            "moves": " ".join(moves)
        }
        if truncateGame(game_data):
            games_data.append(game_data)
            i += 1

    file_postfix = str(file_postfix).replace(".", "_")
    file_path = f"{file_dir}/{agent.id}_agent_vs_stockfish_{file_postfix}.json"
    with open(file_path, "w") as f:
        json.dump(games_data, f, indent=4)
    print(f"[Saved] {file_path}")  # ✅ confirms write succeeded
    return file_path

In [6]:
def overall_similarity_pipeline(json_A, json_B, player_A, player_B):

    print(f"\n--- FULL PIPELINE: {player_A} vs {player_B} ---\n")

    # ============================================================
    # 🔧 FIX: normalize JSON INSIDE PIPELINE (list → string)
    # ============================================================
    def normalize_json(path):
        with open(path, "r") as f:
            games = json.load(f)

        for g in games:
            if isinstance(g.get("moves"), list):
                g["moves"] = " ".join(g["moves"])

        return games

    # Write temporary cleaned versions (no external preprocessing step)
    import tempfile

    def write_temp(games):
        tmp = tempfile.NamedTemporaryFile(delete=False, mode="w", suffix=".json")
        json.dump(games, tmp)
        tmp.close()
        return tmp.name

    clean_A = write_temp(normalize_json(json_A))
    clean_B = write_temp(normalize_json(json_B))

    # ============================================================
    # ORIGINAL PIPELINE (UNCHANGED LOGIC BELOW)
    # ============================================================
    b_A, m_A, l_A = load_json_game_sequences(clean_A, player_A, 1.0)
    b_B, m_B, l_B = load_json_game_sequences(clean_B, player_B, 0.0)

    min_games = min(len(l_A), len(l_B))
    if min_games == 0:
        print("Not enough usable games.")
        return None

    b_A, m_A, l_A = b_A[:min_games], m_A[:min_games], l_A[:min_games]
    b_B, m_B, l_B = b_B[:min_games], m_B[:min_games], l_B[:min_games]

    raw_boards = b_A + b_B
    raw_moves  = m_A + m_B
    raw_labels = l_A + l_B

    combined = list(zip(raw_boards, raw_moves, raw_labels))
    random.shuffle(combined)
    raw_boards, raw_moves, raw_labels = zip(*combined)

    all_boards = np.array(raw_boards)
    all_moves  = np.array(raw_moves)
    all_labels = np.array(raw_labels)

    all_moves_onehot = tf.one_hot(all_moves, NUM_MOVES)

    model = build_style_classifier()
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001, clipnorm=1.0),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1
    )

    model.fit(
        x={"board_seq": all_boards, "move_seq": all_moves_onehot},
        y=all_labels,
        batch_size=32,
        epochs=20,
        validation_split=0.2,
        callbacks=[early_stop],
        verbose=1
    )

    similarity = compute_overall_similarity(
        clean_A, clean_B, player_A, player_B, model
    )

    print(f"Overall playstyle similarity: {similarity:.2f}%")
    os.remove(clean_A)
    os.remove(clean_B)
    return similarity

In [7]:
def hyper_tuning(agent, sf, num_games= 400,file_dir="./data",player_file_dir="./data"):
    a_values = np.linspace(0, 1, 11)[::-1]

    best_a = None
    best_score = float("-inf")

    player_file_path = f"{player_file_dir}/{agent.id}_games.json"

    for a in a_values:
        try:
            agent.a = a

            agent_file_path = simulate_games(
                agent,
                sf,
                num_games,
                file_dir=file_dir,
                file_postfix=f"_a_{a:.2f}"
            )

            score = overall_similarity_pipeline(
                player_file_path,
                agent_file_path,
                f"{agent.id}",
                f"Mimic Agent of {agent.id}"
            )

            print(f"a={a:.2f}, score={score:.3f}")

            if score > best_score:
                best_score = score
                best_a = a

        finally:
            # 🔥 CRITICAL: prevent Colab crashes
            import gc
            tf.keras.backend.clear_session()
            gc.collect()

    print(f"\nBest a: {best_a:.2f} (score={best_score:.3f})")

In [8]:
hyper_tuning(agent,sf,num_games= 400,file_dir="./data400")

/storage/home/jmy5612/model V7/policy.py:197: UserWarning: Note that even though you've set Stockfish to play on a weaker elo or skill level, get_top_moves will still return the top moves of full strength Stockfish.
  top = self.sf.get_top_moves(top_k)
/storage/home/jmy5612/model V7/policy.py:142: UserWarning: Note that even though you've set Stockfish to play on a weaker elo or skill level, get_evaluation will still return full strength Stockfish's evaluation of the position.
  info = self.sf.get_evaluation()


[Saved] ./data400/Bijay_1549_agent_vs_stockfish__a_1_00.json

--- FULL PIPELINE: Bijay_1549 vs Mimic Agent of Bijay_1549 ---

Epoch 1/20
20/20 [==============================] - 22s 828ms/step - loss: 1.7664 - accuracy: 0.6156 - val_loss: 1.6846 - val_accuracy: 0.7134
Epoch 2/20
20/20 [==============================] - 14s 727ms/step - loss: 1.5932 - accuracy: 0.8357 - val_loss: 1.3491 - val_accuracy: 0.9618
Epoch 3/20
20/20 [==============================] - 15s 730ms/step - loss: 1.1664 - accuracy: 0.9697 - val_loss: 1.1162 - val_accuracy: 0.9363
Epoch 4/20
20/20 [==============================] - 15s 730ms/step - loss: 0.9982 - accuracy: 0.9793 - val_loss: 0.9585 - val_accuracy: 0.9809
Epoch 5/20
20/20 [==============================] - 15s 728ms/step - loss: 0.9215 - accuracy: 0.9936 - val_loss: 0.8973 - val_accuracy: 0.9936
Epoch 6/20
20/20 [==============================] - 15s 731ms/step - loss: 0.8815 - accuracy: 0.9920 - val_loss: 0.8530 - val_accuracy: 0.9936
Epoch 7/20
20/20